# 20 — Bronze → Silver
For each table (parents first): pick unprocessed Bronze batches → validate (rules in `_common`) →
reject orphans → de-duplicate → Delta MERGE. Rejected rows go to `silver.quarantine` with the rules they failed.

In [ ]:
%run ./_common

In [ ]:
SILVER_ORDER = ["BANK", "CURRENCY_FX", "PARTY", "LETTER_OF_CREDIT", "LC_PARTY_ROLE", "LC_AMENDMENT", "LC_EXPOSURE_EVENT"]
PROCESSED = fq("silver", "processed_batches")


def pending_batches(table):
    bronze = spark.table(fq("bronze", table))
    done = [r["batch_id"] for r in spark.table(PROCESSED).where(F.col("table_name") == table).collect()]
    return bronze.where(~F.col("batch_id").isin(done)) if done else bronze


def reject_orphans(good, table):
    """Rows whose parent key is not in the (already built) Silver parent table."""
    bad_parts = []
    for col, parent, parent_col in ORPHAN_CHECKS.get(table, []):
        parent_keys = spark.table(fq("silver", parent)).select(F.col(parent_col).alias("_pk")).distinct()
        orphan = good.join(parent_keys, good[col] == parent_keys["_pk"], "left_anti") \
                     .withColumn("_failed", F.array(F.lit(f"orphan_{col}")))
        good = good.join(parent_keys, good[col] == parent_keys["_pk"], "left_semi")
        bad_parts.append(orphan)
    return good, bad_parts


def latest_per_key(df, keys, order_col):
    w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc(), F.col("ingested_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def process(table):
    cfg = TABLES[table]
    todo = pending_batches(table)
    batches = [r["batch_id"] for r in todo.select("batch_id").distinct().collect()]
    if not batches:
        print(f"{table}: no new batches")
        return
    if cfg["kind"] == "F":                       # full refresh: only the newest snapshot matters
        newest = todo.groupBy("batch_id").agg(F.max("ingested_at").alias("t")).orderBy(F.col("t").desc()).first()["batch_id"]
        todo = todo.where(F.col("batch_id") == newest)

    good, bad = split_valid(todo, get_rules(table))
    good, orphan_bad = reject_orphans(good, table)
    bad_parts = [bad] + orphan_bad

    if cfg["kind"] == "I":
        good = latest_per_key(good, cfg["pk"], cfg["order"])
    elif cfg["kind"] == "A":
        good = latest_per_key(good, cfg["pk"], "ingested_at")
        # replayed events: same source_event_ref published under a new event_id; keep the first one
        w = Window.partitionBy("source_event_ref").orderBy(F.col("event_id").asc())
        good = good.withColumn("_rn", F.row_number().over(w))
        dup_in_batch = good.where("_rn > 1").withColumn("_failed", F.array(F.lit("duplicate_source_event_ref")))
        good = good.where("_rn = 1").drop("_rn")
        bad_parts.append(dup_in_batch.drop("_rn"))
        target = fq("silver", table)
        if spark.catalog.tableExists(target):
            known = spark.table(target).select("source_event_ref").distinct()
            dup_vs_silver = good.join(known, "source_event_ref", "left_semi") \
                                .withColumn("_failed", F.array(F.lit("duplicate_source_event_ref")))
            good = good.join(known, "source_event_ref", "left_anti")
            bad_parts.append(dup_vs_silver)
    else:
        good = good.dropDuplicates(cfg["pk"])

    out = (good.select(*cfg["cols"], F.col("batch_id").alias("_source_batch_id"), F.col("ingested_at").alias("_ingested_at"))
               .withColumn("_silver_loaded_at", F.current_timestamp()))
    target = fq("silver", table)

    # Quarantine BEFORE writing `target`. Every entry in bad_parts is a lazy plan, and the
    # append-table duplicate check reads `target` itself; evaluating it after the write would
    # re-read the rows just inserted and flag each one as a duplicate of itself. Day 1 hides
    # this (the table does not exist yet, so the check is skipped) -- it only bites from the
    # second batch onwards, quarantining the entire batch.
    n_bad = 0
    for part in bad_parts:
        c = part.count()
        if c:
            quarantine(part, table)
            n_bad += c

    if cfg["kind"] == "F":
        out.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    elif cfg["kind"] == "I":
        upsert(out, target, cfg["pk"], cfg["order"])
    else:
        insert_only(out, target, cfg["pk"])

    spark.createDataFrame([(table, b) for b in batches], "table_name string, batch_id string") \
         .withColumn("processed_at", F.current_timestamp()).write.format("delta").mode("append").saveAsTable(PROCESSED)
    print(f"{table}: batches={len(batches)} quarantined={n_bad}")


for t in SILVER_ORDER:
    process(t)

In [ ]:
display(spark.table(fq("silver", "quarantine"))
        .select("source_table", F.explode("failed_rules").alias("rule"))
        .groupBy("source_table", "rule").count().orderBy("source_table", "rule"))